# Phase 1 Notebook — Retrieval Basics

This notebook is aligned with the Phase 1 instructions:
- load the dataset with the `json` module;
- inspect fields and compute dataset statistics;
- build a shared `content` field for documents and queries;
- implement the three required retrieval methods: TF-IDF, BM25+, and embeddings;
- evaluate them with `Precision`, `Recall`, `MRR`, and `Accuracy`;
- inspect embeddings in 2D;
- compare models and produce the Kaggle submission.

For the current setup the notebook evaluates retrieval at `stage 0`, so `Accuracy` is fixed to `0.0`.

In [ ]:
from pathlib import Path
import csv
import importlib
import json
import os
import re
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 140)

TOP_K = 100
FINAL_MODEL = 'embedding'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
EMBEDDING_BATCH = 256
TSNE_SAMPLE_SIZE = 2000
RANDOM_STATE = 42
EVAL_STAGE = 0
OUTPUT_FILENAME = 'solutions_SeaFour.csv'
PROJECT_FOLDER_NAME = 'retrieval_project'

# Keep TOP_K=100 for the default submission.
# For the report, study the influence of k later with values such as [10, 20, 50, 100].

TOKEN_RE = re.compile(r'[a-z0-9]+')
RUN_CACHE = {}
EMBEDDING_CACHE = {}


def format_seconds(seconds):
    if seconds < 1:
        return f'{seconds * 1000:.1f} ms'
    if seconds < 60:
        return f'{seconds:.2f} s'
    return f'{seconds / 60:.2f} min'



def log_message(message, level='INFO'):
    timestamp = time.strftime('%H:%M:%S')
    print(f'[{timestamp}] {level:<5} {message}', flush=True)



def log_section(title):
    line = '=' * 88
    print(f'\n{line}\n{title}\n{line}', flush=True)



def ensure_package(module_name, package_name=None):
    package_name = package_name or module_name
    try:
        importlib.import_module(module_name)
        log_message(f'Package ready: {package_name}')
    except ImportError:
        log_message(f'Installing missing package: {package_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])
        log_message(f'Installed: {package_name}')


def resolve_output_path(filename=OUTPUT_FILENAME):
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        output_path = drive_root / PROJECT_FOLDER_NAME / filename
    else:
        output_path = Path.cwd() / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    log_message(f'Submission file will be saved to: {output_path}')
    return output_path



def is_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False



def maybe_mount_google_drive():
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        return

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        print('Google Drive already mounted.')
        return

    print('Google Colab detected. Mounting Google Drive...')
    drive.mount('/content/drive')



def candidate_data_dirs():
    cwd = Path.cwd().resolve()
    candidates = [
        Path('/kaggle/input/retrieval-engine-competition'),
        cwd / 'data',
        cwd.parent / 'data',
        cwd.parent.parent / 'data',
    ]

    for parent in (cwd, *cwd.parents):
        if parent.name == PROJECT_FOLDER_NAME:
            candidates.append(parent / 'data')

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        candidates.append(drive_root / PROJECT_FOLDER_NAME / 'data')
        candidates.append(drive_root / 'Colab Notebooks' / PROJECT_FOLDER_NAME / 'data')
        candidates.extend(path / 'data' for path in drive_root.glob(f'*/{PROJECT_FOLDER_NAME}'))

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        unique_candidates.append(candidate)
    return unique_candidates



def resolve_data_dir():
    maybe_mount_google_drive()
    cwd = Path.cwd().resolve()
    print(f'Current working directory: {cwd}')
    print(f'Running in Colab: {is_colab()}')
    print('Searching for dataset directory...')

    for dirname, _, filenames in os.walk('/kaggle/input'):
        if ('docs.json' in filenames or 'documents.json' in filenames) and 'queries_test.json' in filenames:
            print(f'Using Kaggle dataset directory: {dirname}')
            return Path(dirname)

    searched = []
    for candidate in candidate_data_dirs():
        print(f'Checking data directory candidate: {candidate}')
        searched.append(str(candidate))
        has_docs = (candidate / 'docs.json').exists() or (candidate / 'documents.json').exists()
        has_queries = (candidate / 'queries_test.json').exists()
        if has_docs and has_queries:
            print(f'Using dataset directory: {candidate}')
            return candidate

    raise FileNotFoundError(
        'Could not find the dataset directory. Checked:\n'
        + '\n'.join(searched)
        + '\n\nExpected docs.json or documents.json and queries_test.json under the data/ directory.'
    )



def existing_path(data_dir, names):
    for name in names:
        candidate = data_dir / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find any of: {names}')



def load_json_records(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for key in ['documents', 'docs', 'queries', 'items', 'data']:
            value = raw.get(key)
            if isinstance(value, list):
                return value
        if all(isinstance(v, dict) for v in raw.values()):
            return list(raw.values())
    raise ValueError(f'Unsupported JSON structure in {path}')



def value_to_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)



def create_content_column(df, columns):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''
    merged = []
    for _, row in out[columns].iterrows():
        merged.append(' '.join(value_to_text(row[col]) for col in columns).strip().lower())
    out['content'] = merged
    out['id'] = out['id'].astype(str)
    return out



def tokenize(text):
    txt = str(text or '').lower()
    txt = re.sub(r'[-_/]', ' ', txt)
    return TOKEN_RE.findall(txt)



def token_length_stats(texts):
    lengths = texts.fillna('').map(lambda text: len(tokenize(text)))
    return {
        'min': int(lengths.min()),
        'max': int(lengths.max()),
        'mean': float(lengths.mean()),
        'median': float(lengths.median()),
        'std': float(lengths.std(ddof=0)),
    }



def load_ground_truth(path):
    with open(path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    ground_truth = {}
    total_judgments = 0

    for qid, info in raw.items():
        if isinstance(info, dict):
            items = info.get('relevant_doc_ids', [])
        else:
            items = info

        relevant = set()
        for item in items:
            total_judgments += 1
            if isinstance(item, dict):
                doc_id = item.get('doc_id', item.get('id'))
                relevance = item.get('relevance', item.get('label', 1))
            else:
                doc_id = item
                relevance = 1
            if relevance and doc_id is not None:
                relevant.add(str(doc_id))
        ground_truth[str(qid)] = relevant

    return ground_truth, raw, total_judgments



def describe_available_fields(df):
    rows = []
    for col in df.columns:
        example = value_to_text(df.iloc[0][col]) if len(df) else ''
        rows.append({
            'field': col,
            'dtype': str(df[col].dtype),
            'non_null': int(df[col].notna().sum()),
            'sample': example[:120],
        })
    return pd.DataFrame(rows)



def compute_dataset_statistics(docs_df, train_queries_df, test_queries_df, ground_truth, total_judgments):
    rel_counts = pd.Series([len(ground_truth.get(str(qid), set())) for qid in train_queries_df['id']], dtype='int64')
    category_counts = docs_df['category'].fillna('unknown').astype(str).value_counts().sort_values(ascending=False) if 'category' in docs_df.columns else pd.Series(dtype='int64')
    return {
        'num_documents': int(len(docs_df)),
        'num_train_queries': int(len(train_queries_df)),
        'num_test_queries': int(len(test_queries_df)),
        'num_relevance_judgments': int(total_judgments),
        'num_categories': int(len(category_counts)),
        'category_counts': category_counts,
        'doc_length_stats': token_length_stats(docs_df['content']),
        'train_query_length_stats': token_length_stats(train_queries_df['content']),
        'test_query_length_stats': token_length_stats(test_queries_df['content']),
        'relevant_docs_per_query': {
            'min': int(rel_counts.min()),
            'max': int(rel_counts.max()),
            'mean': float(rel_counts.mean()),
            'median': float(rel_counts.median()),
            'std': float(rel_counts.std(ddof=0)),
        },
    }



def topk_from_scores(scores, top_k):
    top_k = min(top_k, scores.shape[1])
    part = np.argpartition(scores, -top_k, axis=1)[:, -top_k:]
    part_scores = np.take_along_axis(scores, part, axis=1)
    order = np.argsort(part_scores, axis=1)[:, ::-1]
    topk_indices = np.take_along_axis(part, order, axis=1)
    topk_scores = np.take_along_axis(scores, topk_indices, axis=1)
    return topk_indices, topk_scores



def build_run(query_ids, doc_ids, topk_indices, topk_scores, latency_sec):
    return {
        'query_ids': np.asarray(query_ids, dtype=str),
        'doc_ids': np.asarray(doc_ids, dtype=str),
        'topk_indices': topk_indices,
        'topk_scores': topk_scores,
        'topk_doc_ids': np.asarray(doc_ids, dtype=str)[topk_indices],
        'latency_sec': float(latency_sec),
    }



def trim_run(run, top_k):
    top_k = min(int(top_k), run['topk_indices'].shape[1])
    return build_run(run['query_ids'], run['doc_ids'], run['topk_indices'][:, :top_k], run['topk_scores'][:, :top_k], run['latency_sec'])



def get_cached_run(cache_key, top_k):
    cached_runs = RUN_CACHE.get(cache_key, {})
    for cached_top_k in sorted(cached_runs):
        if cached_top_k >= top_k:
            if cached_top_k == top_k:
                return cached_runs[cached_top_k], cached_top_k
            return trim_run(cached_runs[cached_top_k], top_k), cached_top_k
    return None, None



def store_cached_run(cache_key, run):
    RUN_CACHE.setdefault(cache_key, {})[run['topk_indices'].shape[1]] = run
    return run



def show_run_summary(model_name, run, preview_queries=3, preview_docs=5):
    retrieved_k = int(run['topk_indices'].shape[1])
    log_message(
        f'{model_name}: {len(run["query_ids"])} queries x {len(run["doc_ids"])} docs | top_k={retrieved_k} | latency={format_seconds(run["latency_sec"])}'
    )
    rows = []
    for idx in range(min(preview_queries, len(run['query_ids']))):
        rows.append({
            'query_id': run['query_ids'][idx],
            'top_docs': ', '.join(run['topk_doc_ids'][idx][:preview_docs].tolist()),
            'best_score': float(run['topk_scores'][idx][0]) if retrieved_k else np.nan,
        })
    if rows:
        display(pd.DataFrame(rows))



def build_run_overview(runs_by_model):
    rows = []
    for model_name, run in runs_by_model.items():
        rows.append({
            'Model': model_name,
            'Queries': len(run['query_ids']),
            'Documents': len(run['doc_ids']),
            'top_k': int(run['topk_indices'].shape[1]),
            'LatencySec': float(run['latency_sec']),
            'Latency': format_seconds(run['latency_sec']),
        })
    return pd.DataFrame(rows).sort_values('LatencySec').reset_index(drop=True)



def run_tfidf_search(docs_df, queries_df, top_k=100, cache_key=None):
    if cache_key is not None:
        cached_run, cached_top_k = get_cached_run(cache_key, top_k)
        if cached_run is not None:
            source = 'exact cache hit' if cached_top_k == top_k else f'sliced from cached top_k={cached_top_k}'
            log_message(f'TF-IDF cache hit ({source})')
            return cached_run

    log_section(f'TF-IDF retrieval | top_k={top_k}')
    t0 = time.time()
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
    try:
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    except ValueError as err:
        if 'After pruning, no terms remain' not in str(err):
            raise
        log_message('No terms remained with min_df=2, retrying with min_df=1', level='WARN')
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    query_vectors = vectorizer.transform(queries_df['content'])
    log_message(f'TF-IDF vocabulary size: {len(vectorizer.vocabulary_):,}')
    log_message(f'Document matrix shape: {doc_vectors.shape}')
    log_message(f'Query matrix shape: {query_vectors.shape}')
    scores = cosine_similarity(query_vectors, doc_vectors)
    log_message(f'Score matrix shape: {scores.shape}')
    topk_indices, topk_scores = topk_from_scores(scores, top_k)
    run = build_run(queries_df['id'], docs_df['id'], topk_indices, topk_scores, time.time() - t0)
    log_message(f'TF-IDF completed in {format_seconds(run["latency_sec"])}')
    if cache_key is not None:
        store_cached_run(cache_key, run)
    return run



def run_bm25_search(docs_df, queries_df, top_k=100, cache_key=None):
    if cache_key is not None:
        cached_run, cached_top_k = get_cached_run(cache_key, top_k)
        if cached_run is not None:
            source = 'exact cache hit' if cached_top_k == top_k else f'sliced from cached top_k={cached_top_k}'
            log_message(f'BM25 cache hit ({source})')
            return cached_run

    from rank_bm25 import BM25Plus

    log_section(f'BM25+ retrieval | top_k={top_k}')
    t0 = time.time()
    log_message('Tokenizing document corpus...')
    tokenized_docs = [tokenize(text) for text in docs_df['content']]
    avg_doc_len = float(np.mean([len(tokens) for tokens in tokenized_docs])) if tokenized_docs else 0.0
    log_message(f'Tokenized {len(tokenized_docs):,} documents | average length={avg_doc_len:.1f} tokens')
    bm25 = BM25Plus(tokenized_docs)
    log_message('Scoring queries against BM25 index...')
    query_texts = queries_df['content'].tolist()
    progress_step = max(1, len(query_texts) // 5)
    score_rows = []
    for idx, text in enumerate(query_texts, start=1):
        score_rows.append(bm25.get_scores(tokenize(text)))
        if idx == 1 or idx % progress_step == 0 or idx == len(query_texts):
            log_message(f'BM25 progress: {idx}/{len(query_texts)} queries scored')
    scores = np.vstack(score_rows)
    log_message(f'Score matrix shape: {scores.shape}')
    topk_indices, topk_scores = topk_from_scores(scores, top_k)
    run = build_run(queries_df['id'], docs_df['id'], topk_indices, topk_scores, time.time() - t0)
    log_message(f'BM25+ completed in {format_seconds(run["latency_sec"])}')
    if cache_key is not None:
        store_cached_run(cache_key, run)
    return run



def encode_embeddings(docs_df, queries_df, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH, dataset_name='train'):
    cache_key = (dataset_name, model_name, batch_size, len(docs_df), len(queries_df))
    if cache_key in EMBEDDING_CACHE:
        log_message(f'Embedding cache hit for {dataset_name} set with model {model_name}')
        return EMBEDDING_CACHE[cache_key]

    from sentence_transformers import SentenceTransformer

    log_message(f'Loading embedding model: {model_name}')
    model = SentenceTransformer(model_name)
    log_message(f'Encoding {len(docs_df):,} documents with batch_size={batch_size}')
    doc_embeddings = model.encode(
        docs_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    doc_embeddings = np.asarray(doc_embeddings)
    log_message(f'Document embeddings ready: {doc_embeddings.shape}')
    log_message(f'Encoding {len(queries_df):,} {dataset_name} queries')
    query_embeddings = model.encode(
        queries_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    query_embeddings = np.asarray(query_embeddings)
    log_message(f'Query embeddings ready: {query_embeddings.shape}')
    EMBEDDING_CACHE[cache_key] = (doc_embeddings, query_embeddings)
    return EMBEDDING_CACHE[cache_key]



def run_embedding_search(docs_df, queries_df, top_k=100, model_name=EMBEDDING_MODEL, batch_size=EMBEDDING_BATCH, cache_key=None, dataset_name='train'):
    if cache_key is not None:
        cached_run, cached_top_k = get_cached_run(cache_key, top_k)
        if cached_run is not None:
            source = 'exact cache hit' if cached_top_k == top_k else f'sliced from cached top_k={cached_top_k}'
            log_message(f'Embedding retrieval cache hit ({source})')
            doc_embeddings, query_embeddings = encode_embeddings(docs_df, queries_df, model_name=model_name, batch_size=batch_size, dataset_name=dataset_name)
            return cached_run, doc_embeddings, query_embeddings

    log_section(f'Embedding retrieval ({dataset_name}) | top_k={top_k}')
    t0 = time.time()
    doc_embeddings, query_embeddings = encode_embeddings(
        docs_df,
        queries_df,
        model_name=model_name,
        batch_size=batch_size,
        dataset_name=dataset_name,
    )
    log_message(f'Computing dense similarity matrix: {query_embeddings.shape} @ {doc_embeddings.shape}')
    scores = query_embeddings @ doc_embeddings.T
    log_message(f'Score matrix shape: {scores.shape}')
    topk_indices, topk_scores = topk_from_scores(scores, top_k)
    run = build_run(queries_df['id'], docs_df['id'], topk_indices, topk_scores, time.time() - t0)
    log_message(f'Embedding retrieval completed in {format_seconds(run["latency_sec"])}')
    if cache_key is not None:
        store_cached_run(cache_key, run)
    return run, doc_embeddings, query_embeddings



def precision_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        retrieved = [str(doc_id) for doc_id in doc_ids[idx][:k]]
        hits = sum(1 for doc_id in retrieved if doc_id in relevant)
        values.append(hits / len(retrieved) if retrieved else 0.0)
    return float(np.mean(values)) if values else 0.0



def recall_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        if not relevant:
            values.append(0.0)
            continue
        retrieved = {str(doc_id) for doc_id in doc_ids[idx][:k]}
        values.append(len(relevant & retrieved) / len(relevant))
    return float(np.mean(values)) if values else 0.0



def mrr_at_k(run, ground_truth, k=None):
    doc_ids = run['topk_doc_ids']
    query_ids = run['query_ids']
    k = doc_ids.shape[1] if k is None else k
    values = []
    for idx, qid in enumerate(query_ids):
        relevant = ground_truth.get(str(qid), set())
        rr = 0.0
        for rank, doc_id in enumerate(doc_ids[idx][:k], start=1):
            if str(doc_id) in relevant:
                rr = 1.0 / rank
                break
        values.append(rr)
    return float(np.mean(values)) if values else 0.0



def accuracy_for_stage(stage=EVAL_STAGE):
    if stage == 0:
        return 0.0
    raise NotImplementedError('Accuracy is only defined as 0.0 for stage 0 in this notebook.')



def compute_metrics(run, ground_truth, k, stage=EVAL_STAGE):
    precision = precision_at_k(run, ground_truth, k=k)
    recall = recall_at_k(run, ground_truth, k=k)
    mrr = mrr_at_k(run, ground_truth, k=k)
    accuracy = accuracy_for_stage(stage=stage)
    return {
        'Precision': precision,
        'Recall': recall,
        'MRR': mrr,
        'Accuracy': accuracy,
        'LatencySec': float(run['latency_sec']),
    }



def evaluate_run(run, ground_truth, k, stage=EVAL_STAGE):
    metrics = compute_metrics(run, ground_truth, k=k, stage=stage)
    return {
        f'Precision@{k}': metrics['Precision'],
        f'Recall@{k}': metrics['Recall'],
        f'MRR@{k}': metrics['MRR'],
        'Accuracy': metrics['Accuracy'],
        'LatencySec': metrics['LatencySec'],
    }



def evaluate_runs(runs_by_model, ground_truth, k, stage=EVAL_STAGE):
    rows = {}
    log_section(f'Retrieval evaluation @ k={k} | stage={stage}')
    for model_name, run in runs_by_model.items():
        row = evaluate_run(run, ground_truth, k=k, stage=stage)
        rows[model_name] = row
        log_message(
            f'{model_name}: MRR@{k}={row[f"MRR@{k}"]:.4f} | Accuracy={row["Accuracy"]:.4f} | latency={format_seconds(run["latency_sec"])}'
        )
    eval_df = pd.DataFrame(rows).T
    eval_df.index.name = 'Model'
    return eval_df.sort_values(by=[f'MRR@{k}', f'Recall@{k}', f'Precision@{k}'], ascending=False)



def compare_k_values(runs_by_model, ground_truth, k_values, stage=EVAL_STAGE):
    k_values = sorted({int(k) for k in k_values})
    if not k_values:
        raise ValueError('k_values must contain at least one positive integer.')

    max_available_k = min(run['topk_indices'].shape[1] for run in runs_by_model.values())
    if max(k_values) > max_available_k:
        raise ValueError(
            f'Requested k up to {max(k_values)}, but current runs only store top-{max_available_k}. '
            'Increase TOP_K and rerun the model cells first.'
        )

    log_section(f'Influence of k | stage={stage}')
    rows = []
    for k in k_values:
        log_message(f'Computing metrics for k={k}')
        for model_name, run in runs_by_model.items():
            metrics = compute_metrics(run, ground_truth, k=k, stage=stage)
            rows.append({'Model': model_name, 'k': k, **metrics})
            log_message(
                f'k={k:>3} | {model_name:<9} MRR={metrics["MRR"]:.4f} | Accuracy={metrics["Accuracy"]:.4f}'
            )

    k_study_df = pd.DataFrame(rows).sort_values(['k', 'MRR', 'Recall', 'Precision'], ascending=[True, False, False, False]).reset_index(drop=True)
    k_summary_df = k_study_df.pivot(index='k', columns='Model', values='MRR').sort_index()
    return k_study_df, k_summary_df



def sample_indices_for_tsne(docs_df, sample_size=TSNE_SAMPLE_SIZE, random_state=RANDOM_STATE):
    sample_size = min(sample_size, len(docs_df))
    rng = np.random.default_rng(random_state)
    if 'category' not in docs_df.columns or docs_df['category'].isna().all():
        return np.sort(rng.choice(len(docs_df), size=sample_size, replace=False))
    groups = docs_df['category'].fillna('unknown').astype(str)
    categories = groups.unique().tolist()
    per_category = max(1, sample_size // max(1, len(categories)))
    sampled = []
    for category in categories:
        idx = np.flatnonzero(groups.to_numpy() == category)
        take = min(per_category, len(idx))
        if take:
            sampled.extend(rng.choice(idx, size=take, replace=False).tolist())
    sampled = sorted(set(sampled))
    if len(sampled) < sample_size:
        remaining = np.setdiff1d(np.arange(len(docs_df)), np.asarray(sampled, dtype=int), assume_unique=False)
        extra = min(sample_size - len(sampled), len(remaining))
        if extra:
            sampled.extend(rng.choice(remaining, size=extra, replace=False).tolist())
    return np.asarray(sorted(sampled[:sample_size]), dtype=int)



def plot_embedding_projection(docs_df, doc_embeddings, query_embeddings, sample_size=TSNE_SAMPLE_SIZE, random_state=RANDOM_STATE):
    log_section('Embedding inspection (t-SNE)')
    sampled_doc_indices = sample_indices_for_tsne(docs_df, sample_size=sample_size, random_state=random_state)
    log_message(f'Projecting {len(sampled_doc_indices):,} sampled documents and {len(query_embeddings):,} queries')
    log_message('Running t-SNE. This can take a while...')
    combined = np.vstack([doc_embeddings[sampled_doc_indices], query_embeddings])
    projection = TSNE(n_components=2, perplexity=30, random_state=random_state, init='pca', learning_rate='auto').fit_transform(combined)
    doc_projection = projection[:len(sampled_doc_indices)]
    query_projection = projection[len(sampled_doc_indices):]
    log_message('t-SNE projection finished')

    plt.figure(figsize=(10, 7))
    if 'category' in docs_df.columns:
        categories = docs_df.iloc[sampled_doc_indices]['category'].fillna('unknown').astype(str)
        for category in sorted(categories.unique()):
            mask = (categories == category).to_numpy()
            plt.scatter(doc_projection[mask, 0], doc_projection[mask, 1], s=12, alpha=0.55, label=f'doc:{category}')
    else:
        plt.scatter(doc_projection[:, 0], doc_projection[:, 1], s=12, alpha=0.5, label='documents')
    plt.scatter(query_projection[:, 0], query_projection[:, 1], s=40, c='black', marker='x', linewidths=1.0, label='train queries')
    plt.title('t-SNE projection of document and query embeddings')
    plt.xlabel('t-SNE dim 1')
    plt.ylabel('t-SNE dim 2')
    plt.legend(loc='best', fontsize=8)
    plt.tight_layout()
    return doc_projection, query_projection



def write_kaggle_submission(run, sample_csv_path, output_csv_path):
    pred_map = {str(qid): [str(doc_id) for doc_id in run['topk_doc_ids'][idx].tolist()] for idx, qid in enumerate(run['query_ids'])}
    with open(sample_csv_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError('Invalid sample submission format.')
    id_col = fieldnames[0]
    pred_col = fieldnames[1]
    category_col = fieldnames[2] if len(fieldnames) >= 3 else None
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            qid = str(row[id_col])
            if qid not in pred_map:
                raise ValueError(f'Missing prediction for query_id: {qid}')
            out_row = {id_col: qid, pred_col: json.dumps(pred_map[qid])}
            if category_col is not None:
                out_row[category_col] = row.get(category_col, '?') or '?'
            writer.writerow(out_row)



DATA_DIR = resolve_data_dir()
DOCS_PATH = existing_path(DATA_DIR, ['docs.json', 'documents.json'])
TRAIN_QUERIES_PATH = existing_path(DATA_DIR, ['queries_train.json'])
TEST_QUERIES_PATH = existing_path(DATA_DIR, ['queries_test.json'])
QRELS_PATH = existing_path(DATA_DIR, ['qgts_train.json', 'gts.json'])
SUBMISSION_TEMPLATE_PATH = existing_path(DATA_DIR, ['submission.csv'])
OUTPUT_PATH = resolve_output_path(OUTPUT_FILENAME)

print(f'Data directory: {DATA_DIR}')
print(f'Output path: {OUTPUT_PATH}')
print(f'Final submission model: {FINAL_MODEL}')


## Load And Prepare The Data

The project instructions explicitly ask to load the dataset with the `json` module. The raw JSON records are loaded below and then converted to pandas DataFrames for easier analysis.

The `content` field is created by concatenating the available `title`, `text`, and `tags` fields. This gives each retrieval model the same textual representation.

In [ ]:
docs_raw = load_json_records(DOCS_PATH)
train_queries_raw = load_json_records(TRAIN_QUERIES_PATH)
test_queries_raw = load_json_records(TEST_QUERIES_PATH)
ground_truth, qrels_raw, total_judgments = load_ground_truth(QRELS_PATH)

docs_df = pd.DataFrame(docs_raw)
train_queries_df = pd.DataFrame(train_queries_raw)
test_queries_df = pd.DataFrame(test_queries_raw)

docs_df = create_content_column(docs_df, ['title', 'text', 'tags'])
train_queries_df = create_content_column(train_queries_df, ['title', 'text', 'tags'])
test_queries_df = create_content_column(test_queries_df, ['title', 'text', 'tags'])

print(f'Documents             : {len(docs_df):,}')
print(f'Train queries         : {len(train_queries_df):,}')
print(f'Test queries          : {len(test_queries_df):,}')
print(f'Relevance judgments   : {total_judgments:,}')

display(docs_df.head(2))
display(train_queries_df.head(2))

## Dataset Inspection

This section addresses the first part of Phase 1: understanding what the dataset looks like, which fields it contains, how long documents and queries are, how many categories exist, and how many relevant documents appear per training query.

In [ ]:
doc_fields_df = describe_available_fields(docs_df)
train_query_fields_df = describe_available_fields(train_queries_df)
dataset_stats = compute_dataset_statistics(docs_df, train_queries_df, test_queries_df, ground_truth, total_judgments)

summary_df = pd.DataFrame([
    {'statistic': 'num_documents', 'value': dataset_stats['num_documents']},
    {'statistic': 'num_train_queries', 'value': dataset_stats['num_train_queries']},
    {'statistic': 'num_test_queries', 'value': dataset_stats['num_test_queries']},
    {'statistic': 'num_relevance_judgments', 'value': dataset_stats['num_relevance_judgments']},
    {'statistic': 'num_categories', 'value': dataset_stats['num_categories']},
])

length_stats_df = pd.DataFrame([
    {'group': 'documents', **dataset_stats['doc_length_stats']},
    {'group': 'train_queries', **dataset_stats['train_query_length_stats']},
    {'group': 'test_queries', **dataset_stats['test_query_length_stats']},
    {'group': 'relevant_docs_per_query', **dataset_stats['relevant_docs_per_query']},
])

print('Document fields')
display(doc_fields_df)
print('Train query fields')
display(train_query_fields_df)
print('High-level statistics')
display(summary_df)
print('Length and relevance statistics')
display(length_stats_df)

if dataset_stats['num_categories']:
    category_counts_df = dataset_stats['category_counts'].rename_axis('category').reset_index(name='count')
    print('Document categories')
    display(category_counts_df)

## Dataset Conclusions

Use this section in the final report to explain what the statistics suggest. Typical points to discuss are:
- whether documents are much longer than queries;
- whether relevance labels are sparse or dense;
- whether some categories dominate the corpus;
- why these properties may help lexical models or semantic embeddings.

## TF-IDF Retrieval

TF-IDF stands for Term Frequency–Inverse Document Frequency. It gives higher weight to terms that occur frequently inside a document but are relatively rare in the whole corpus. This helps highlight words that are informative for that document. After vectorizing documents and queries, cosine similarity is used to rank the documents for each query.

The required outputs for this method are `topk_indices_tfidf` and `topk_scores_tfidf`.

## BM25+ Retrieval

BM25+ is a probabilistic retrieval model designed for ranking documents. Compared with plain TF-IDF, it includes term-frequency saturation and document-length normalization, so repeated occurrences of a term help only up to a point and long documents are not unfairly favored. BM25+ is often a strong lexical baseline for retrieval.

The required outputs for this method are `topk_indices_bm25` and `topk_scores_bm25`.

## Representation Learning With Embeddings

High-dimensional embeddings convert text into dense vectors that capture semantic meaning. Their purpose is to place texts with similar meaning close to each other in vector space, even if they do not share the same exact words. This helps information retrieval because embeddings can handle synonyms, paraphrases, and related concepts better than purely lexical methods.

The required outputs for this method are `topk_indices_embedding` and `topk_scores_embedding`.

In [ ]:
ensure_package('rank_bm25')


In [ ]:
tfidf_run = run_tfidf_search(docs_df, train_queries_df, top_k=TOP_K, cache_key='tfidf_train')
topk_indices_tfidf = tfidf_run['topk_indices']
topk_scores_tfidf = tfidf_run['topk_scores']
show_run_summary('TF-IDF', tfidf_run)


In [ ]:
bm25_run = run_bm25_search(docs_df, train_queries_df, top_k=TOP_K, cache_key='bm25_train')
topk_indices_bm25 = bm25_run['topk_indices']
topk_scores_bm25 = bm25_run['topk_scores']
show_run_summary('BM25+', bm25_run)


In [ ]:
embedding_run, doc_embeddings, query_embeddings = run_embedding_search(
    docs_df,
    train_queries_df,
    top_k=TOP_K,
    model_name=EMBEDDING_MODEL,
    batch_size=EMBEDDING_BATCH,
    cache_key='embedding_train',
    dataset_name='train',
)
topk_indices_embedding = embedding_run['topk_indices']
topk_scores_embedding = embedding_run['topk_scores']
show_run_summary('Embedding', embedding_run)


In [ ]:
train_runs = {
    'tfidf': tfidf_run,
    'bm25': bm25_run,
    'embedding': embedding_run,
}

run_overview_df = build_run_overview(train_runs)
display(run_overview_df)

print(f'topk_indices_tfidf shape     : {topk_indices_tfidf.shape}')
print(f'topk_scores_tfidf shape      : {topk_scores_tfidf.shape}')
print(f'topk_indices_bm25 shape      : {topk_indices_bm25.shape}')
print(f'topk_scores_bm25 shape       : {topk_scores_bm25.shape}')
print(f'topk_indices_embedding shape : {topk_indices_embedding.shape}')
print(f'topk_scores_embedding shape  : {topk_scores_embedding.shape}')


## Embedding Inspection

The assignment asks to inspect the structure of embeddings and visualize them in 2D. The code below prints the embedding shapes and draws a t-SNE projection. If categories are available, document points are colored by category to make clustering easier to interpret.

In [ ]:
log_section('Embedding vectors overview')
print(f'Document embeddings shape: {doc_embeddings.shape}')
print(f'Query embeddings shape   : {query_embeddings.shape}')

plot_embedding_projection(docs_df, doc_embeddings, query_embeddings)


## Embedding Interpretation

After running the t-SNE projection, describe whether you observe clustering or separation. A reasonable explanation is that documents from similar categories or topics tend to be embedded closer together, while queries often lie near documents that share semantic meaning.

## Retrieval Evaluation Suite

The required evaluation metrics are:
- `Precision@k`
- `Recall@k`
- `MRR@k`
- `Accuracy`

This notebook evaluates the current pipeline at `stage 0`, so `Accuracy` is reported as `0.0` for every model.

In [ ]:
train_runs = {
    'tfidf': tfidf_run,
    'bm25': bm25_run,
    'embedding': embedding_run,
}

eval_df = evaluate_runs(train_runs, ground_truth, k=TOP_K)
display(eval_df)


## Compare The Retrieval Models

Use the table above to answer the Phase 1 comparison questions in your report:
- Which model has the best precision, recall, and MRR?
- Why might lexical models do well on some queries and embeddings on others?
- What are the trade-offs between retrieval quality and latency?
- How does the value of `k` affect each metric?

The table is sorted by `MRR@k`, then `Recall@k`, then `Precision@k`. `Accuracy` stays at `0.0` because this notebook is evaluated at stage 0.

## Influence Of `k`

The `k` study below is now executable as-is. It reuses the already computed retrieval runs, so comparing `k` values is immediate as long as `TOP_K` is at least as large as the biggest value you want to test.

If you want to compare more values, change `K_VALUES` and rerun only that cell.

In [ ]:
DEFAULT_K_VALUES = [10, 20, 50, 100]
K_VALUES = [k for k in DEFAULT_K_VALUES if k <= TOP_K]
if TOP_K not in K_VALUES:
    K_VALUES.append(TOP_K)
K_VALUES = sorted(set(K_VALUES))

train_runs = {
    'tfidf': tfidf_run,
    'bm25': bm25_run,
    'embedding': embedding_run,
}

k_study_df, k_summary_df = compare_k_values(train_runs, ground_truth, k_values=K_VALUES)
display(k_study_df)
display(k_summary_df)


## Final Kaggle Submission

The submission cell below keeps `embedding` as the final submission model, while TF-IDF and BM25+ remain in the notebook for the required comparison.

In [ ]:
if FINAL_MODEL != 'embedding':
    raise ValueError('This notebook is configured to generate the final submission with the embedding model.')

test_embedding_run, _, _ = run_embedding_search(
    docs_df,
    test_queries_df,
    top_k=TOP_K,
    model_name=EMBEDDING_MODEL,
    batch_size=EMBEDDING_BATCH,
    cache_key='embedding_test',
    dataset_name='test',
)
write_kaggle_submission(test_embedding_run, SUBMISSION_TEMPLATE_PATH, OUTPUT_PATH)
show_run_summary('Embedding test submission run', test_embedding_run, preview_queries=2)
print(f'Saved: {OUTPUT_PATH.resolve()}')


In [ ]:
submission_preview = pd.read_csv(OUTPUT_PATH)
print(f'Rows: {len(submission_preview)}, Columns: {list(submission_preview.columns)}')
submission_preview.head()